In [31]:
MLFLOW_TRACKING_URI = '../models/mlruns'
MLFLOW_RUN_ID = "493ac2925d734815b116ac8b5c5f4be9"
EXP_ID = "377147707337886246"
LOG_DATA_PKL    =  "data.pkl"
LOG_MODEL_PKL   =  "model.pkl"
LOG_METRICS_PKL =  "metrics.pkl"
EXPERIMENT_ID = "1"
CLUSTERS_YAML_PATH = "../data/processed/features_skills_clusters_description.yaml"

In [22]:
from urllib.parse import urlparse


In [5]:
import os 
import sklearn
import pickle
import yaml

import pandas as pd

import mlflow
from mlflow.tracking import MlflowClient

## Mlflow

In [6]:
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
client  = MlflowClient()

run = mlflow.get_run(MLFLOW_RUN_ID)
artifacts_path = run.info.artifact_uri

C:\Users\user\AppData\Local\Programs\Python\Python311\Lib\site-packages\mlflow\tracking\_tracking_service\utils.py:177: FutureWarning: The filesystem tracking backend (e.g., './mlruns') will be deprecated in February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://github.com/mlflow/mlflow/issues/18534 for more details and migration guidance.
  return FileStore(store_uri, store_uri)


## Load Model

In [32]:
artifact_path = os.path.join(MLFLOW_TRACKING_URI.replace("file://",""),
                            EXP_ID,
                             MLFLOW_RUN_ID,
                             "artifacts"
                            )

In [37]:
#load model
model_path = os.path.join(artifact_path, LOG_MODEL_PKL) 
with open(model_path, "rb") as f:
    model_pkl = pickle.load(f)
model = model_pkl["model_object"]
model

,steps,"[('standardscaler', ...), ('featureunion', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,transformer_list,"[('linear_pca', ...), ('kernel_pca', ...)]"
,n_jobs,None
,transformer_weights,None
,verbose,False


In [33]:
#load data
data_path = os.path.join(artifact_path, LOG_DATA_PKL) 
with open(data_path, "rb") as handle:
    data = pickle.load(handle)
data.keys()

dict_keys(['data_path', 'training_indices', 'test_indices', 'features_names', 'targets_names'])

## Predict Sample Entry

In [40]:
with open(CLUSTERS_YAML_PATH, "rb") as stream:
    clusters_config = yaml.safe_load(stream)
clusters_config

{'skills_group_0': ['PHP',
  'MariaDB',
  'MySQL',
  'SQLite',
  'Drupal',
  'Laravel',
  'Symfony',
  'Vue.js'],
 'skills_group_1': ['Scala',
  'Cassandra',
  'Couchbase',
  'Apache Spark',
  'Hadoop'],
 'skills_group_10': ['TypeScript', 'Angular', 'Angular.js', 'Cordova'],
 'skills_group_11': ['Dart', 'Firebase', 'Flutter'],
 'skills_group_12': ['Ruby', 'Ruby on Rails'],
 'skills_group_13': ['Haskell', 'Julia', 'Rust'],
 'skills_group_14': ['Go', 'Elasticsearch', 'PostgreSQL', 'Redis'],
 'skills_group_15': ['Unity 3D', 'Unreal Engine'],
 'skills_group_16': ['Objective-C', 'Swift'],
 'skills_group_17': ['Assembly', 'C', 'C++'],
 'skills_group_18': ['Chef', 'Puppet'],
 'skills_group_2': ['Bash/Shell/PowerShell',
  'Perl',
  'Python',
  'Django',
  'Flask'],
 'skills_group_3': ['C#', 'VBA', 'Microsoft SQL Server', 'ASP.NET', '.NET'],
 'skills_group_4': ['Java', 'Kotlin', 'IBM DB2', 'Oracle', 'Spring'],
 'skills_group_5': ['MongoDB',
  'Express',
  'Gatsby',
  'React.js',
  'Node.js',
  

In [41]:
molten_clusters = [(cluster_name, cluster_skill)
                   for cluster_name, cluster_skills in clusters_config.items()
                   for cluster_skill in cluster_skills]

clusters_df = pd.DataFrame(molten_clusters, columns=["cluster_name", "skill"])
clusters_df

,cluster_name,skill
0,skills_group_0,PHP
1,skills_group_0,MariaDB
2,skills_group_0,MySQL
3,skills_group_0,SQLite
4,skills_group_0,Drupal
...,...,...
69,skills_group_8,Xamarin
70,skills_group_9,HTML/CSS
71,skills_group_9,JavaScript
72,skills_group_9,SQL


## Recreate clusters feature

In [44]:
sample_skills = ['Scala', 'Hadoop', 'Python']


In [47]:
sample_clusters = clusters_df.copy()

In [48]:
sample_clusters["sample_skills"] = sample_clusters["skill"].isin(sample_skills)

In [49]:
cluster_features = sample_clusters.groupby("cluster_name")["sample_skills"].sum()

In [50]:
cluster_features

cluster_name
skills_group_0     0
skills_group_1     2
skills_group_10    0
skills_group_11    0
skills_group_12    0
skills_group_13    0
skills_group_14    0
skills_group_15    0
skills_group_16    0
skills_group_17    0
skills_group_18    0
skills_group_2     1
skills_group_3     0
skills_group_4     0
skills_group_5     0
skills_group_6     0
skills_group_7     0
skills_group_8     0
skills_group_9     0
Name: sample_skills, dtype: int64

## Create One Hot Encoded skills

In [56]:
feature_names = pd.Series(data["features_names"])

In [57]:
skills_names = feature_names[~feature_names.isin(cluster_features.index)]
skills_names

0                  Assembly
1     Bash/Shell/PowerShell
2                         C
3                        C#
4                       C++
              ...          
69                 Teraform
70            Torch/PyTorch
71                 Unity 3D
72            Unreal Engine
73                  Xamarin
Length: 74, dtype: object

In [58]:
ohe_skills = pd.Series(skills_names.isin(sample_skills).astype(int).tolist(),
                      index = skills_names)

In [59]:
ohe_skills

Assembly                 0
Bash/Shell/PowerShell    0
C                        0
C#                       0
C++                      0
                        ..
Teraform                 0
Torch/PyTorch            0
Unity 3D                 0
Unreal Engine            0
Xamarin                  0
Length: 74, dtype: int64

## Combine features

In [60]:
features = pd.concat([ohe_skills,cluster_features])

In [62]:
features = features[data["features_names"]]

In [63]:
features

Assembly                 0
Bash/Shell/PowerShell    0
C                        0
C#                       0
C++                      0
                        ..
skills_group_5           0
skills_group_6           0
skills_group_7           0
skills_group_8           0
skills_group_9           0
Length: 93, dtype: int64

## Predict

In [65]:
predictions = model.predict_proba([features.values])
positive_probs = [prob[0][1] for prob in predictions]
pd.Series(positive_probs,
         index = data["targets_names"]).sort_values(ascending=False)

Engineer, data                                   0.61
Developer, back-end                              0.56
Data scientist or machine learning specialist    0.24
Data or business analyst                         0.13
Developer, desktop or enterprise applications    0.10
Academic researcher                              0.09
Scientist                                        0.09
Developer, full-stack                            0.08
Developer, QA or test                            0.05
Database administrator                           0.02
DevOps specialist                                0.02
Developer, front-end                             0.02
Developer, game or graphics                      0.02
Developer, embedded applications or devices      0.01
Developer, mobile                                0.01
System administrator                             0.01
dtype: float64